In [8]:
import cv2
from matplotlib import pyplot as plt
import numpy as np

In [9]:
# Load the image
image_path = "..\\dataset\\images\\IMG-20250210-WA0006.jpg"
image = cv2.imread(image_path)
  
# OpenCV opens images as BRG 
# but we want it as RGB We'll 
# also need a grayscale version
img_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [10]:
# Convert the image to HSV color space
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

In [11]:
# Define color range to filter out the polythene bag (assumed to be white/transparent)
lower_bound = np.array([0, 0, 150])  # Adjust for brightness
upper_bound = np.array([180, 50, 255])  # Upper range for white

In [12]:
# Create a mask to remove the bag (white areas)
mask = cv2.inRange(hsv, lower_bound, upper_bound)

# Invert mask to keep objects inside the bag
mask = cv2.bitwise_not(mask)

In [13]:
# Apply morphological operations to clean up small noise
kernel = np.ones((5, 5), np.uint8)
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

In [14]:
# Apply the mask to the original image
filtered = cv2.bitwise_and(image, image, mask=mask)

In [15]:
# Convert the filtered image to grayscale
gray = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY)

In [16]:
# Apply GaussianBlur to reduce noise
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

In [17]:
# Use Canny edge detection
edges = cv2.Canny(blurred, 50, 150)

In [18]:
# Find contours
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [19]:
# Create a copy of the original image to draw contours
output_image = image.copy()

In [20]:
# Loop through all contours and isolate objects inside the bag
for i, contour in enumerate(contours):
    area = cv2.contourArea(contour)
    
    # Ignore very large contours (likely the bag itself) and very small ones (noise)
    if 100 < area < 50000:  
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output_image, (x, y), (x+w, y+h), (0, 255, 0), 2)

In [21]:
# Display the original image with bounding boxes
cv2.imshow('Detected Objects Inside Bag', output_image)

In [22]:
# Save the output image with bounding boxes
cv2.imwrite('filtered_detected_objects.jpg', output_image)

cv2.waitKey(0)
cv2.destroyAllWindows()